In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [2]:
from collections import Counter

In [3]:
file_path = "data/notes.pdf"

cleaning the repeated lines

In [4]:
def detect_repeated_lines(docs, threshold=0.6):
    """
    Lines appearing in more than `threshold` % of pages 
    are likely headers/footers
    """
    all_lines = []
    for doc in docs:
        lines = [line.strip() for line in doc.page_content.split('\n') 
                 if line.strip()]
        all_lines.extend(lines)
    
    total_pages = len(docs)
    line_counts = Counter(all_lines)
    
    # Flag lines appearing in 60%+ of pages
    repeated = {
        line for line, count in line_counts.items() 
        if count / total_pages >= threshold and len(line) > 3
    }
    
    return repeated

def clean_docs(docs):
    repeated_lines = detect_repeated_lines(docs)
    
    print("Detected repeating lines (will be removed):")
    for line in repeated_lines:
        print(f"  → '{line}'")
    
    for doc in docs:
        lines = doc.page_content.split('\n')
        cleaned = [l for l in lines if l.strip() not in repeated_lines]
        doc.page_content = '\n'.join(cleaned)
    
    return docs

In [5]:
loader = PyPDFLoader(file_path)
docs = loader.load()
docs = clean_docs(docs)

Detected repeating lines (will be removed):
  → 'EPCHE, Department of BCA'
  → 'Cyber Security[SEC-4]'


In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
chunks = splitter.split_documents(docs)

In [20]:
print(chunks[0].page_content)

1 
 
 
 
Module-I: Introduction to Cyber security 
Defining Cyberspace and Overview of Computer and Web -technology, Architecture of 
cyberspace, Communication and web technology, Internet, World wide web, Advent of 
internet, Internet infrastructure for data transfer and governance, Internet society, 
Regulation of cyberspace, Concept of cyber security, Issues and challenges of cyber 
security. 
 
Defining Cyberspace 
▪ The term Cyberspace was first coined by William Gibson in the year 1984. 
▪ Cyberspace is the environment in which communication over computer networks occurs.


In [24]:
len(chunks)

53

# Embeddings

In [10]:
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(
    documents=docs, 
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1538.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
query = "By whom the term cyberspace is coined?"

In [21]:
results_with_scores = db.similarity_search_with_score(query, k=3)

for doc, score in results_with_scores:
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print('---')


Score: 0.6130
 1 
 
 
 
Module-I: Introduction to Cyber security 
Defining Cyberspace and Overview of Computer and Web -technology, Architecture of 
cyberspace, Communication and web technology, Internet, World wide web, Advent of 
internet, Internet infrastructure for data transfer and governance, Internet society, 
Regulation of cyberspace, Concept of cyber security, Issues and challenges of cyber 
security. 
 
Defining Cyberspace 
▪ The term Cyberspace was first coined by William Gibson in the year 1984. 
▪ Cyberspace is the environment in which communication over computer networks occurs. 
▪ Cyberspace is the virtual and dynamic space created by the machine clones. Cyberspace 
mainly refers to the computer which is a virtual network and is a medium electronically 
designed to help online communications to occur. 
▪ The primary purpose of creating cyberspace is to share information and communicate 
across the globe. 
▪ Cyberspace is that space in which users share information, inter

Checking model limit

In [22]:
from sentence_transformers import SentenceTransformer

# To get the value of the max sequence_length, we will query the underlying `SentenceTransformer` object used in the RecursiveCharacterTextSplitter
print(
    f"Model's maximum sequence length: {SentenceTransformer('all-MiniLM-L6-v2').max_seq_length}"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4276.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model's maximum sequence length: 256


# Retrevier

In [ ]:
from langchain_classic.chains import RetrievalQA



llm = OpenAI()
retriever = db.as_retriever()

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # This combines all docs into one prompt
    retriever=retriever
)

# Run the chain
response = qa_chain.invoke(query)

ImportError: cannot import name 'OpenAI' from 'langchain_openapi' (c:\dev\python\rag\rag_venv\Lib\site-packages\langchain_openapi\__init__.py)